In [1]:
from google.colab import files
uploaded = files.upload()

Saving Multilabeled sentiment and emotion detection dataset.xlsx to Multilabeled sentiment and emotion detection dataset.xlsx


In [2]:
!pip install transformers torch pandas numpy scikit-learn tqdm
!pip install git+https://github.com/csebuetnlp/normalizer.git

  Cloning https://github.com/csebuetnlp/normalizer.git to /tmp/pip-req-build-lulakrji
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer.git /tmp/pip-req-build-lulakrji
  Resolved https://github.com/csebuetnlp/normalizer.git to commit d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 18.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for normalizer: filename=normalizer-0.0.1-py3-none-any.whl size=6860 sha256=062196042b5a689602649319144ccbd81eb26879eaa284c3894c1c9a7ded8f6f
  Stored in directory: /tmp/pip-ephem-wheel-cache-qrjam8tw/wheels/e3/dc/f8/e8edcf904fe41142198fbc5939cec440d554ea36a80b0631da
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186456 sha256=7529e674d914e4532bd88d581583

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_excel('Multilabeled sentiment and emotion detection dataset.xlsx', sheet_name='Sheet1')
print(f"Dataset shape: {df.shape}")
print(df.head())

# Encode labels
sentiment_encoder = LabelEncoder()
emotion_encoder = LabelEncoder()

df['sentiment_label'] = sentiment_encoder.fit_transform(df['Sentiment'])
df['emotion_label'] = emotion_encoder.fit_transform(df['Emotion'])

print("\nSentiment classes:", sentiment_encoder.classes_)
print("Emotion classes:", emotion_encoder.classes_)
print("\nSentiment distribution:")
print(df['sentiment_label'].value_counts())
print("\nEmotion distribution:")
print(df['emotion_label'].value_counts())

Dataset shape: (40911, 3)
                                                Text      Sentiment    Emotion
0  এসব পাছা কবিরের আজাইররা পোস্ট। হেতি ক, "যদি ভু...       Negative     Sexual
1  জুতা মারি গরুদান মারান!! আপনার আম্মা আব্বার খে...  Very Negative    Fearful
2  সুখে থাকতে ভুতে কিলাই, প্রবাদটা এর কাজের দ্বার...       Negative    Hateful
3  তর মাথায় ব্রেন আছে সেটা তুই দেখছস? না দেখলে ভা...  Very Negative    Hateful
4                                 আপনি কি নাস্তিক..?       Negative  Religious

Sentiment classes: ['Negative' 'Neutral ' 'Positive' 'Very Negative' 'Very Positive']
Emotion classes: ['Calm' 'Fearful' 'Happy' 'Hateful' 'Religious' 'Sexual' 'Surprised']

Sentiment distribution:
sentiment_label
0    10757
3     8979
1     8662
2     7286
4     5227
Name: count, dtype: int64

Emotion distribution:
emotion_label
2    8154
0    7583
5    7250
1    7086
3    4919
6    3470
4    2449
Name: count, dtype: int64


In [4]:
print("Unique emotions in dataset:", df['Emotion'].unique())
print("Number of unique emotions:", len(df['Emotion'].unique()))
print("Emotion encoder classes:", emotion_encoder.classes_)
print("Min emotion label:", df['emotion_label'].min())
print("Max emotion label:", df['emotion_label'].max())

Unique emotions in dataset: ['Sexual' 'Fearful' 'Hateful' 'Religious' 'Calm' 'Happy' 'Surprised']
Number of unique emotions: 7
Emotion encoder classes: ['Calm' 'Fearful' 'Happy' 'Hateful' 'Religious' 'Sexual' 'Surprised']
Min emotion label: 0
Max emotion label: 6


In [5]:
print(f"📊 Total Dataset Size: {len(df)}")
print(f"\n📈 Sentiment Distribution:")
print(df['Sentiment'].value_counts())
print(f"\n🎭 Emotion Distribution:")
print(df['Emotion'].value_counts())

📊 Total Dataset Size: 40911

📈 Sentiment Distribution:
Sentiment
Negative         10757
Very Negative     8979
Neutral           8662
Positive          7286
Very Positive     5227
Name: count, dtype: int64

🎭 Emotion Distribution:
Emotion
Happy        8154
Calm         7583
Sexual       7250
Fearful      7086
Hateful      4919
Surprised    3470
Religious    2449
Name: count, dtype: int64


In [6]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


In [7]:
# ============================================
# STEP 2: CREATE DATASET AND DATALOADERS (FIXED)
# ============================================
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔧 Device: {device}")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")
print("✅ Tokenizer loaded")

# Define custom dataset (FIXED VERSION)
class BanglaSentimentEmotionDataset(Dataset):
    def __init__(self, texts, sentiment_labels, emotion_labels, tokenizer, max_len=128):
        self.texts = texts
        self.sentiment_labels = sentiment_labels
        self.emotion_labels = emotion_labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        sentiment_label = self.sentiment_labels[item]
        emotion_label = self.emotion_labels[item]

        # New method without encode_plus
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'sentiment_labels': torch.tensor(sentiment_label, dtype=torch.long),
            'emotion_labels': torch.tensor(emotion_label, dtype=torch.long)
        }

# Split data into train and validation sets
train_df, val_df = train_test_split(
    df, test_size=0.1,
    random_state=42,
    stratify=df[['sentiment_label', 'emotion_label']]
)

# Create dataset instances
train_dataset = BanglaSentimentEmotionDataset(
    texts=train_df['Text'].to_numpy(),
    sentiment_labels=train_df['sentiment_label'].to_numpy(),
    emotion_labels=train_df['emotion_label'].to_numpy(),
    tokenizer=tokenizer
)

val_dataset = BanglaSentimentEmotionDataset(
    texts=val_df['Text'].to_numpy(),
    sentiment_labels=val_df['sentiment_label'].to_numpy(),
    emotion_labels=val_df['emotion_label'].to_numpy(),
    tokenizer=tokenizer
)

# Create DataLoader instances
batch_size = 16
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"\nTrain dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


🔧 Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Tokenizer loaded

Train dataset size: 36819
Validation dataset size: 4092
Train batches: 2302
Validation batches: 256


In [8]:
# STEP 3: DEFINE MODEL
# ============================================
import torch.nn as nn
from transformers import AutoModel

class BanglaSentimentEmotionModel(nn.Module):
    def __init__(self, num_sentiments=5, num_emotions=7, dropout=0.1):
        super().__init__()
        self.banglabert = AutoModel.from_pretrained("csebuetnlp/banglabert")
        hidden_size = self.banglabert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.sentiment_head = nn.Linear(hidden_size, num_sentiments)
        self.emotion_head = nn.Linear(hidden_size, num_emotions)

    def forward(self, input_ids, attention_mask):
        outputs = self.banglabert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        pooled = self.dropout(pooled)
        return self.sentiment_head(pooled), self.emotion_head(pooled)

# Initialize model
model = BanglaSentimentEmotionModel(
    num_sentiments=5,
    num_emotions=7
).to(device)

print(f"\n✅ Model loaded on {device}")
print(f"📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Sentiment: 5 classes")
print(f"   Emotion: 7 classes")


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Model loaded on cuda
📊 Total parameters: 110,035,980
   Sentiment: 5 classes
   Emotion: 7 classes


In [9]:
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm

# Verify labels before training
print("\n🔍 Verifying labels in dataloaders:")
for batch in train_loader:
    print(f"Sentiment labels - min: {batch['sentiment_labels'].min()}, max: {batch['sentiment_labels'].max()}")
    print(f"Emotion labels - min: {batch['emotion_labels'].min()}, max: {batch['emotion_labels'].max()}")
    assert batch['emotion_labels'].max() < 7, f"Found emotion label {batch['emotion_labels'].max()}"
    assert batch['sentiment_labels'].max() < 5, f"Found sentiment label {batch['sentiment_labels'].max()}"
    print("✅ Label verification passed!")
    break

# Training loop
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 5

for epoch in range(num_epochs):
    # Training phase
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sentiment_labels = batch['sentiment_labels'].to(device)
        emotion_labels = batch['emotion_labels'].to(device)

        # Forward pass
        sentiment_logits, emotion_logits = model(input_ids, attention_mask)

        # Calculate loss
        sentiment_loss = nn.CrossEntropyLoss()(sentiment_logits, sentiment_labels)
        emotion_loss = nn.CrossEntropyLoss()(emotion_logits, emotion_labels)
        loss = sentiment_loss + emotion_loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"📉 Epoch {epoch+1} - Average Training Loss: {avg_train_loss:.4f}")

    # Validation phase
    model.eval()
    val_loss = 0
    val_sentiment_correct = 0
    val_emotion_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sentiment_labels = batch['sentiment_labels'].to(device)
            emotion_labels = batch['emotion_labels'].to(device)

            sentiment_logits, emotion_logits = model(input_ids, attention_mask)

            sentiment_loss = nn.CrossEntropyLoss()(sentiment_logits, sentiment_labels)
            emotion_loss = nn.CrossEntropyLoss()(emotion_logits, emotion_labels)
            loss = sentiment_loss + emotion_loss
            val_loss += loss.item()

            # Calculate accuracy
            sentiment_preds = torch.argmax(sentiment_logits, dim=1)
            emotion_preds = torch.argmax(emotion_logits, dim=1)
            val_sentiment_correct += (sentiment_preds == sentiment_labels).sum().item()
            val_emotion_correct += (emotion_preds == emotion_labels).sum().item()
            total_samples += sentiment_labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    sentiment_acc = val_sentiment_correct / total_samples
    emotion_acc = val_emotion_correct / total_samples

    print(f"📊 Epoch {epoch+1} - Validation Loss: {avg_val_loss:.4f}")
    print(f"🎯 Sentiment Accuracy: {sentiment_acc:.4f} ({sentiment_acc*100:.2f}%)")
    print(f"🎭 Emotion Accuracy: {emotion_acc:.4f} ({emotion_acc*100:.2f}%)")
    print("-" * 60)

print("\n✅ Training Completed Successfully!")


🔍 Verifying labels in dataloaders:
Sentiment labels - min: 0, max: 4
Emotion labels - min: 0, max: 5
✅ Label verification passed!


Epoch 1/5 [Train]:   0%|          | 0/2302 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Epoch 1/5 [Train]: 100%|██████████| 2302/2302 [15:34<00:00,  2.46it/s]


📉 Epoch 1 - Average Training Loss: 1.1862


...

📊 Epoch 4 - Validation Loss: 0.0259
🎯 Sentiment Accuracy: 0.9929 (99.29%)
🎭 Emotion Accuracy: 0.9985 (99.85%)
------------------------------------------------------------


Epoch 5/5 [Train]: 100%|██████████| 2302/2302 [15:34<00:00,  2.46it/s]


📉 Epoch 5 - Average Training Loss: 0.0588


Epoch 5/5 [Val]: 100%|██████████| 256/256 [00:35<00:00,  7.30it/s]

📊 Epoch 5 - Validation Loss: 0.0312
🎯 Sentiment Accuracy: 0.9914 (99.14%)
🎭 Emotion Accuracy: 0.9971 (99.71%)
------------------------------------------------------------

✅ Training Completed Successfully!
